# Entrenamiento en Kaggle

Antes de correr:
1. **Add Input** → busca `moltean/fruits` (Fruits-360) y agrégalo.
2. **Settings** → Accelerator: **GPU T4 x2** o **P100**; Internet: **On** (requiere teléfono verificado).
3. Para correr sin tener la pestaña abierta: **Save Version → Save & Run All**.

Este notebook solo clona el repo y llama a `src/`. Los cambios de código van en el repo, no aquí.

In [ ]:
!git clone -q https://github.com/11909/Clasificador-fruta-verdura.git repo
%cd repo
!pip install -q timm

Localiza las carpetas `Training` y `Test` del dataset adjunto y genera la config de esta corrida.

In [ ]:
import glob, yaml

train_dirs = sorted(glob.glob("/kaggle/input/**/Training", recursive=True))
test_dirs = sorted(glob.glob("/kaggle/input/**/Test", recursive=True))
print("train:", train_dirs)
print("test:", test_dirs)

cfg = yaml.safe_load(open("configs/baseline.yaml"))
cfg["data"]["train_dirs"] = train_dirs[:1]   # ajusta si adjuntas más de un dataset
cfg["data"]["test_dirs"] = test_dirs[:1]
cfg["data"]["max_samples"] = None            # p. ej. 2000 para una prueba rápida
yaml.safe_dump(cfg, open("configs/run.yaml", "w"), allow_unicode=True)

Revisa que todas las clases tengan etiqueta en `data/mapping.csv`.

In [ ]:
!python scripts/check_mapping.py {train_dirs[0]} | tail -5

Opcional: seguimiento con Weights & Biases. Guarda tu API key en **Add-ons → Secrets** con el nombre `WANDB_API_KEY`.

In [ ]:
USAR_WANDB = False

if USAR_WANDB:
    import os
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    cfg["wandb"]["enabled"] = True
    yaml.safe_dump(cfg, open("configs/run.yaml", "w"), allow_unicode=True)
    !pip install -q wandb

In [ ]:
!python -m src.train --config configs/run.yaml

El mejor modelo queda en `/kaggle/working/outputs/best.pt` y las métricas de prueba en `test_metrics.json`.

In [ ]:
!cat /kaggle/working/outputs/test_metrics.json